# 07 — GNN Graph Generation

This notebook converts the AqSolDB molecules into graph objects suitable for the molecular GNN.

The graph representation combines:

- **Nodes:** atomic numbers, chemical atom features and atom-centred G6 density/curvature features.
- **Edges:** covalent connectivity, 3D neighbourhoods, bond attributes, distances, radial basis functions and radial kernels.
- **Coordinates:** the selected RDKit conformer positions.
- **Target:** experimental aqueous solubility (`Y`, log mol/L).

The official TDC `train_val` and `test` partitions are kept separate. The train/validation scaffold split will be created later in the training notebook.

## Important decisions

- Explicit hydrogens are retained, as in the conformer-generation workflow in `03_feature_engineering.ipynb`.
- The conformer and graph objects are cached to disk, so expensive 3D and G6 calculations do not need to be repeated.
- Graphs are saved **before normalization**. The normalizer must be fitted later using only the training fold of each scaffold split.
- No molecular-level descriptors are repeated on the graph nodes.

## 1. Environment and project paths

Run this notebook from the repository's `notebooks/` directory or from the repository root. The project root is detected automatically.

PyTorch and PyTorch Geometric must be installed in the active environment. The PyG installation command depends on the installed PyTorch and CUDA versions, so install it using the official PyG instructions rather than hard-coding a CUDA-specific wheel here.

In [1]:
from pathlib import Path
import json
import sys

import numpy as np
import pandas as pd
import torch
from tqdm.auto import tqdm

from rdkit import Chem, RDLogger
from rdkit.Chem import AllChem

import torch_geometric
from torch_geometric.loader import DataLoader

RDLogger.DisableLog("rdApp.warning")

CURRENT_DIR = Path.cwd().resolve()
PROJECT_ROOT = CURRENT_DIR.parent if CURRENT_DIR.name == "notebooks" else CURRENT_DIR

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

DATA_RAW_DIR = PROJECT_ROOT / "data" / "raw"
DATA_RAW_DIR.mkdir(parents=True, exist_ok=True)
PROCESSED_DIR = PROJECT_ROOT / "data" / "processed" / "aqsoldb"
GRAPH_DIR = PROCESSED_DIR / "gnn"
CONFORMER_CACHE_DIR = GRAPH_DIR / "conformers"

GRAPH_DIR.mkdir(parents=True, exist_ok=True)
CONFORMER_CACHE_DIR.mkdir(parents=True, exist_ok=True)

print("Project root:", PROJECT_ROOT)
print("PyTorch:", torch.__version__)
print("PyTorch Geometric:", torch_geometric.__version__)
print("RDKit:", Chem.rdBase.rdkitVersion)

Project root: C:\Users\alexc\OneDrive\Desktop\proyectos\admet-ml
PyTorch: 2.15.0+cpu
PyTorch Geometric: 2.8.1
RDKit: 2023.09.6


c:\Users\alexc\AppData\Local\Programs\Python\Python311\Lib\site-packages\torch\jit\_script.py:1491: FutureWarning: `torch.jit.script` is deprecated. Please switch to `torch.compile` or `torch.export`.
  warnings.warn(


## 2. Import the project modules

The following modules should already exist under `src/gnn/`:

- `config.py`
- `atom_features.py`
- `edge_features.py`
- `density_features.py`
- `graph_builder.py`

The density module reuses the G6 implementation in `src/features/dg_gl_features.py`.

In [2]:
from tdc.benchmark_group import admet_group

from src.features import conformers
from src.gnn.config import GNNConfig
from src.gnn.graph_builder import build_graph, get_graph_feature_names

## 3. GNN configuration

The graph-generation configuration is centralized in `GNNConfig`. Changing the graph cutoff, edge mode, G6 kernel, element channels, density types or gradient thresholds changes the graph representation and therefore requires regenerating the graph cache.

In [3]:
BENCHMARK_NAME = "Solubility_AqSolDB"

config = GNNConfig()

# Conformer settings match the workflow used in 03_feature_engineering.ipynb.
N_CONFORMERS = 5
CONFORMER_RANDOM_SEED = 42
CONFORMER_PRUNE_RMS = 0.5

# Set to True only when you intentionally want to regenerate the 3D cache.
REBUILD_CONFORMERS = False

# Set to True only when you intentionally want to rebuild graph tensors.
REBUILD_GRAPHS = False

# Intermediate graph checkpoints make long graph-generation runs resumable.
GRAPH_CHECKPOINT_EVERY = 100

print(config)
print("Node features:", len(get_graph_feature_names(config)["x"]))
print("  Chemical:", len(get_graph_feature_names(config)["x_chem"]))
print("  G6:", len(get_graph_feature_names(config)["x_dg"]))
print("Edge features:", len(get_graph_feature_names(config)["edge_attr"]))

GNNConfig(graph=GraphConfig(edge_mode='hybrid', radius_cutoff_angstrom=5.0, max_neighbors=None, rbf_num_centers=32, include_bond_attributes=True, include_distances=True), atomic_features=AtomicFeaturesConfig(include_basic_chemistry=True, include_density=True, include_curvature=True, elements=('H', 'C', 'N', 'O', 'S', 'P', 'F', 'Cl', 'Br', 'I', 'Na'), other_label='Other', density_types=('number',), kernel='exponential', kernel_configs=((2.5, 1.3),), sigma_angstrom=0.0, gradient_epsilon=1e-08, relative_gradient_epsilon=1e-06, include_validity_masks=True, geometry_transform='signed_log1p', normalization='train_standard'), model=ModelConfig(hidden_dim=128, num_message_passing_layers=4, message_mlp_hidden_dim=128, dropout=0.15, residual_connections=True, pooling='mean', readout_hidden_dim=128), training=TrainingConfig(batch_size=32, max_epochs=200, learning_rate=0.001, weight_decay=0.0001, loss='huber', huber_delta=1.0, early_stopping_patience=25, gradient_clip_norm=5.0, random_seed=42, num

## 4. Load the official TDC partitions

We use the same benchmark and partition names as the existing acquisition and feature-engineering notebooks. We do not create a new random split here.

In [4]:
group = admet_group(path=str(DATA_RAW_DIR))
benchmark = group.get(BENCHMARK_NAME)

train_val = benchmark["train_val"].copy().reset_index(drop=True)
test = benchmark["test"].copy().reset_index(drop=True)

print("Benchmark:", benchmark["name"])
print("Train + validation:", train_val.shape)
print("Test:", test.shape)
print("Columns:", train_val.columns.tolist())

train_val.head()

Found local copy...


Benchmark: solubility_aqsoldb
Train + validation: (7985, 3)
Test: (1997, 3)
Columns: ['Drug_ID', 'Drug', 'Y']


,Drug_ID,Drug,Y
0,4-chlorobenzaldehyde,O=Cc1ccc(Cl)cc1,-2.177078
1,vinyltoluene,C=Cc1cccc(C)c1,-3.123150
2,4-(dimethylamino)benzaldehyde,CN(C)c1ccc(C=O)cc1,-2.282769
3,2-methyl-1-phenylpropan-2-yl acetate,CC(=O)OC(C)(C)Cc1ccccc1,-2.394650
4,5-methoxy-1-[4-(trifluoromethyl)phenyl]pentan-...,COCCCCC(=O)c1ccc(C(F)(F)F)cc1,-3.544060


## 5. Generate and cache 3D conformers

`03_feature_engineering.ipynb` generates five ETKDG conformers per molecule, optimizes them with MMFF when available (otherwise UFF), and selects the lowest-energy converged conformer. Explicit hydrogens are retained.

That notebook currently keeps `ROMol3D` in memory but does not persist the RDKit molecules. Therefore, unless a conformer cache already exists, this section must generate them once again. It saves the resulting RDKit molecules and selected conformer IDs as pickle files; subsequent runs load those files instead of regenerating conformers.

In [5]:
def prepare_3d_molecule(mol):
    """Replicate the conformer-selection procedure used in notebook 03."""
    try:
        mol_3d, conf_ids = conformers.generate_conformers(
            mol,
            n_conformers=N_CONFORMERS,
            random_seed=CONFORMER_RANDOM_SEED,
            prune_rms_thresh=CONFORMER_PRUNE_RMS,
        )

        if AllChem.MMFFHasAllMoleculeParams(mol_3d):
            results = conformers.optimize_conformers(
                mol_3d,
                force_field="MMFF",
                max_iters=200,
                num_threads=0,
            )
            method = "MMFF"

        elif AllChem.UFFHasAllMoleculeParams(mol_3d):
            results = conformers.optimize_conformers(
                mol_3d,
                force_field="UFF",
                max_iters=200,
                num_threads=0,
            )
            method = "UFF"

        else:
            return mol_3d, conf_ids[0], "ETKDG_fallback"

        converged = [
            (idx, energy)
            for idx, (status, energy) in enumerate(results)
            if status == 0
        ]

        if converged:
            lowest_idx, _ = min(converged, key=lambda item: item[1])
            conf_id = conf_ids[lowest_idx]
            return mol_3d, conf_id, method

        return mol_3d, conf_ids[0], "ETKDG_fallback"

    except (ValueError, RuntimeError):
        return None, None, "embedding_failed"

In [6]:
def load_or_build_conformer_cache(split_df, split_name):
    """Load cached RDKit conformers or generate and persist them once."""
    cache_path = CONFORMER_CACHE_DIR / (
        f"{split_name}_3d_n{N_CONFORMERS}_seed{CONFORMER_RANDOM_SEED}_rms{str(CONFORMER_PRUNE_RMS).replace('.', 'p')}.pkl"
    )

    if cache_path.exists() and not REBUILD_CONFORMERS:
        cached = pd.read_pickle(cache_path)
        same_smiles = (
            cached["Drug"].astype(str).tolist()
            == split_df["Drug"].astype(str).tolist()
        )
        same_targets = np.allclose(
            cached["Y"].to_numpy(dtype=float),
            split_df["Y"].to_numpy(dtype=float),
            equal_nan=True,
        )
        if same_smiles and same_targets:
            print(f"Loaded conformer cache: {cache_path.name}")
            return cached

    records = []

    for row_id, row in tqdm(
        split_df.iterrows(),
        total=len(split_df),
        desc=f"Generating {split_name} conformers",
    ):
        mol = Chem.MolFromSmiles(row["Drug"])

        if mol is None:
            mol_3d, conf_id, method = None, None, "invalid_smiles"
        else:
            mol_3d, conf_id, method = prepare_3d_molecule(mol)

        records.append({
            "row_id": int(row_id),
            "Drug": row["Drug"],
            "Y": float(row["Y"]),
            "ROMol3D": mol_3d,
            "ConformerId": conf_id,
            "conformer_method": method,
        })

    conformer_df = pd.DataFrame(records)
    conformer_df.to_pickle(cache_path)
    print(f"Saved conformer cache: {cache_path}")

    return conformer_df

In [7]:
train_val_conformers = load_or_build_conformer_cache(train_val, "train_val")
test_conformers = load_or_build_conformer_cache(test, "test")

for split_name, frame in [
    ("train_val", train_val_conformers),
    ("test", test_conformers),
]:
    valid = frame["ROMol3D"].notna() & frame["ConformerId"].notna()
    print(
        f"{split_name}: {valid.sum()} valid conformers / {len(frame)} "
        f"({100 * valid.mean():.2f}%)"
    )
    print(frame.loc[~valid, "conformer_method"].value_counts())

Generating train_val conformers:   0%|          | 0/7985 [00:00<?, ?it/s]

[16:21:50] UFFTYPER: Unrecognized charge state for atom: 0
[16:21:50] UFFTYPER: Unrecognized atom type: Zn+2 (0)
[16:21:50] UFFTYPER: Unrecognized charge state for atom: 20
[16:21:50] UFFTYPER: Unrecognized atom type: Zn+2 (20)
[16:21:55] UFFTYPER: Unrecognized charge state for atom: 0
[16:21:55] UFFTYPER: Unrecognized atom type: Pb3+3 (0)
[16:21:55] UFFTYPER: Unrecognized charge state for atom: 17
[16:21:55] UFFTYPER: Unrecognized atom type: Pb3+3 (17)
[16:21:59] UFFTYPER: Unrecognized charge state for atom: 1
[16:21:59] UFFTYPER: Unrecognized charge state for atom: 1
[16:21:59] UFFTYPER: Unrecognized charge state for atom: 1
[16:21:59] UFFTYPER: Unrecognized charge state for atom: 1
[16:22:17] UFFTYPER: Unrecognized charge state for atom: 11
[16:22:17] UFFTYPER: Unrecognized charge state for atom: 11
[16:22:17] UFFTYPER: Unrecognized charge state for atom: 0
[16:22:17] UFFTYPER: Unrecognized atom type: Zn+2 (0)
[16:22:17] UFFTYPER: Unrecognized charge state for atom: 0
[16:22:17] UFF

Saved conformer cache: C:\Users\alexc\OneDrive\Desktop\proyectos\admet-ml\data\processed\aqsoldb\gnn\conformers\train_val_3d_n5_seed42_rms0p5.pkl


Generating test conformers:   0%|          | 0/1997 [00:00<?, ?it/s]

[16:28:18] UFFTYPER: Unrecognized atom type: Ca+2 (0)
[16:28:29] UFFTYPER: Unrecognized atom type: Cr2+3 (0)
[16:28:29] UFFTYPER: Unrecognized atom type: Cr2+3 (0)
[16:28:29] UFFTYPER: Unrecognized atom type: Cr2+3 (93)
[16:28:29] UFFTYPER: Unrecognized atom type: Cr2+3 (94)
[16:28:29] UFFTYPER: Unrecognized atom type: Cu6+1 (25)
[16:28:29] UFFTYPER: Unrecognized atom type: Cu6+1 (25)
[16:28:29] UFFTYPER: Unrecognized atom type: Cu6+1 (19)
[16:28:29] UFFTYPER: Unrecognized atom type: Cu6+1 (19)
[16:29:08] UFFTYPER: Unrecognized atom type: Ca+2 (0)
[16:29:25] UFFTYPER: Unrecognized atom type: Rh3 (0)
[16:29:25] UFFTYPER: Unrecognized atom type: Rh3 (9)
[16:29:30] UFFTYPER: Unrecognized atom type: Cr1+3 (0)
[16:29:30] UFFTYPER: Unrecognized atom type: Cr1+3 (31)
[16:29:31] UFFTYPER: Unrecognized atom type: Ca+2 (0)
[16:29:37] UFFTYPER: Unrecognized charge state for atom: 3
[16:29:37] UFFTYPER: Unrecognized charge state for atom: 3
[16:29:37] UFFTYPER: Unrecognized charge state for atom: 

Saved conformer cache: C:\Users\alexc\OneDrive\Desktop\proyectos\admet-ml\data\processed\aqsoldb\gnn\conformers\test_3d_n5_seed42_rms0p5.pkl
train_val: 7961 valid conformers / 7985 (99.70%)
conformer_method
embedding_failed    24
Name: count, dtype: int64
test: 1991 valid conformers / 1997 (99.70%)
conformer_method
embedding_failed    6
Name: count, dtype: int64


## 6. Inspect the conformer convention

The graph must use the same atom ordering and the same explicit-hydrogen policy for node features, G6 features, edge features and coordinates. We therefore pass the exact same `ROMol3D` object and `ConformerId` to all feature-generation functions.

In [8]:
valid_sample = train_val_conformers.loc[
    train_val_conformers["ROMol3D"].notna()
    & train_val_conformers["ConformerId"].notna()
].iloc[0]

sample_mol = valid_sample["ROMol3D"]
sample_conf_id = int(valid_sample["ConformerId"])
sample_conf = sample_mol.GetConformer(sample_conf_id)

n_explicit_h = sum(atom.GetAtomicNum() == 1 for atom in sample_mol.GetAtoms())

print("SMILES:", valid_sample["Drug"])
print("Conformer ID:", sample_conf_id)
print("Atoms including H:", sample_mol.GetNumAtoms())
print("Explicit hydrogen atoms:", n_explicit_h)
print("Conformer is 3D:", sample_conf.Is3D())

SMILES: O=Cc1ccc(Cl)cc1
Conformer ID: 0
Atoms including H: 14
Explicit hydrogen atoms: 5
Conformer is 3D: True


## 7. Smoke test: build a few molecular graphs

Before processing the full dataset, build a small sample. This verifies that the feature modules agree on atom ordering and feature dimensions, and that the selected conformer can be used by the graph builder.

In [9]:
N_SAMPLE_GRAPHS = 5

sample_rows = (
    train_val_conformers.loc[
        train_val_conformers["ROMol3D"].notna()
        & train_val_conformers["ConformerId"].notna()
    ]
    .head(N_SAMPLE_GRAPHS)
)

sample_graphs = []

for row in tqdm(sample_rows.itertuples(index=False), total=len(sample_rows)):
    graph = build_graph(
        mol=row.ROMol3D,
        conf_id=int(row.ConformerId),
        config=config,
        y=float(row.Y),
        smiles=row.Drug,
        mol_id=f"train_val:{row.row_id}",
    )
    graph.row_id = int(row.row_id)
    sample_graphs.append(graph)

print("Sample graphs:", len(sample_graphs))

  0%|          | 0/5 [00:00<?, ?it/s]

Sample graphs: 5


In [10]:
sample_graph = sample_graphs[0]
feature_names = get_graph_feature_names(config)

print(sample_graph)
print("\nNode tensors")
print("z:", tuple(sample_graph.z.shape))
print("x_chem:", tuple(sample_graph.x_chem.shape))
print("x_dg:", tuple(sample_graph.x_dg.shape))
print("x:", tuple(sample_graph.x.shape))
print("pos:", tuple(sample_graph.pos.shape))

print("\nEdge tensors")
print("edge_index:", tuple(sample_graph.edge_index.shape))
print("edge_attr:", tuple(sample_graph.edge_attr.shape))
print("Target:", tuple(sample_graph.y.shape))

print("\nFirst chemical feature names:", feature_names["x_chem"])
print("Number of G6 features:", len(feature_names["x_dg"]))
print("First edge feature names:", feature_names["edge_attr"][:12])

Data(x=[14, 89], edge_index=[2, 170], edge_attr=[170, 43], pos=[14, 3], z=[14], x_chem=[14, 17], x_dg=[14, 72], num_nodes=14, y=[1], smiles='O=Cc1ccc(Cl)cc1', mol_id='train_val:0', conf_id=[1], row_id=0)

Node tensors
z: (14,)
x_chem: (14, 17)
x_dg: (14, 72)
x: (14, 89)
pos: (14, 3)

Edge tensors
edge_index: (2, 170)
edge_attr: (170, 43)
Target: (1,)

First chemical feature names: ['formal_charge', 'heavy_atom_degree', 'total_attached_hydrogens', 'atomic_mass', 'is_aromatic', 'is_in_ring', 'hybridization_sp', 'hybridization_sp2', 'hybridization_sp3', 'hybridization_sp3d', 'hybridization_sp3d2', 'hybridization_unspecified', 'hybridization_other', 'chirality_unassigned', 'chirality_r', 'chirality_s', 'chirality_other']
Number of G6 features: 72
First edge feature names: ['is_bonded', 'bond_type_single', 'bond_type_double', 'bond_type_triple', 'bond_type_aromatic', 'bond_type_other', 'bond_is_aromatic', 'bond_is_conjugated', 'bond_is_in_ring', 'distance_angstrom', 'gaussian_rbf_00', 'gaus

In [11]:
# Check that node tensors share the same atom dimension and edge tensors align.
assert sample_graph.z.size(0) == sample_graph.x.size(0)
assert sample_graph.x_chem.size(0) == sample_graph.x_dg.size(0)
assert sample_graph.pos.size(0) == sample_graph.x.size(0)
assert sample_graph.edge_index.size(0) == 2
assert sample_graph.edge_attr.size(0) == sample_graph.edge_index.size(1)
assert torch.isfinite(sample_graph.x).all()
assert torch.isfinite(sample_graph.edge_attr).all()
assert torch.isfinite(sample_graph.pos).all()

print("Single-graph consistency checks passed.")

Single-graph consistency checks passed.


## 8. Build and cache the complete graph datasets

The full graph generation can take time because each molecule requires atom-centred G6 calculations. The function below:

- skips molecules without a valid conformer;
- records graph-generation failures instead of silently discarding them;
- saves intermediate checkpoints every `GRAPH_CHECKPOINT_EVERY` processed molecules;
- resumes from a compatible partial checkpoint if execution is interrupted;
- reuses a completed graph cache when the configuration and feature schema match.

`train_val` and `test` remain separate. No normalization is applied here.

In [12]:
def _json_config(config):
    """Convert the dataclass configuration to JSON-compatible values."""
    from dataclasses import asdict
    return json.loads(json.dumps(asdict(config), sort_keys=True))


def _graph_manifest(split_name):
    names = get_graph_feature_names(config)
    return {
        "split": split_name,
        "config": _json_config(config),
        "node_feature_names": names["x"],
        "edge_feature_names": names["edge_attr"],
        "conformer_settings": {
            "n_conformers": N_CONFORMERS,
            "random_seed": CONFORMER_RANDOM_SEED,
            "prune_rms": CONFORMER_PRUNE_RMS,
        },
    }


def _save_partial_graphs(graphs, records, paths, manifest):
    graph_path, records_path, manifest_path = paths
    torch.save(graphs, graph_path)
    pd.DataFrame(records).to_csv(records_path, index=False)
    manifest_path.write_text(json.dumps(manifest, indent=2))


def build_graphs_for_split(conformer_df, split_name):
    """Build, checkpoint, cache and return one TDC partition's graphs."""
    graph_path = GRAPH_DIR / f"{split_name}_graphs.pt"
    metadata_path = GRAPH_DIR / f"{split_name}_graph_metadata.csv"
    failures_path = GRAPH_DIR / f"{split_name}_graph_failures.csv"
    manifest_path = GRAPH_DIR / f"{split_name}_graph_manifest.json"

    partial_graph_path = GRAPH_DIR / f"{split_name}_graphs.partial.pt"
    partial_records_path = GRAPH_DIR / f"{split_name}_graphs.partial.csv"
    partial_manifest_path = GRAPH_DIR / f"{split_name}_graphs.partial.json"

    manifest = _graph_manifest(split_name)

    if (
        graph_path.exists()
        and metadata_path.exists()
        and manifest_path.exists()
        and not REBUILD_GRAPHS
    ):
        saved_manifest = json.loads(manifest_path.read_text())
        if saved_manifest == manifest:
            print(f"Loading graph cache: {graph_path.name}")
            graphs = torch.load(
                graph_path,
                map_location="cpu",
                weights_only=False,
            )
            metadata = pd.read_csv(metadata_path)
            failures = (
                pd.read_csv(failures_path)
                if failures_path.exists()
                else pd.DataFrame()
            )
            return graphs, metadata, failures

    valid_df = conformer_df.loc[
        conformer_df["ROMol3D"].notna()
        & conformer_df["ConformerId"].notna()
    ].copy()

    partial_paths = (
        partial_graph_path,
        partial_records_path,
        partial_manifest_path,
    )

    if (
        partial_graph_path.exists()
        and partial_records_path.exists()
        and partial_manifest_path.exists()
        and not REBUILD_GRAPHS
        and json.loads(partial_manifest_path.read_text()) == manifest
    ):
        print(f"Resuming partial graph cache for {split_name}.")
        graphs = torch.load(
            partial_graph_path,
            map_location="cpu",
            weights_only=False,
        )
        records = pd.read_csv(partial_records_path).to_dict("records")
    else:
        graphs = []
        records = []

    processed_row_ids = {int(record["row_id"]) for record in records}
    pending = valid_df.loc[
        ~valid_df["row_id"].isin(processed_row_ids)
    ]

    for processed_count, row in enumerate(
        tqdm(
            pending.itertuples(index=False),
            total=len(pending),
            desc=f"Building {split_name} graphs",
        ),
        start=1,
    ):
        try:
            graph = build_graph(
                mol=row.ROMol3D,
                conf_id=int(row.ConformerId),
                config=config,
                y=float(row.Y),
                smiles=row.Drug,
                mol_id=f"{split_name}:{row.row_id}",
            )
            graph.row_id = int(row.row_id)
            graph.dataset_split = split_name
            graphs.append(graph)

            records.append({
                "row_id": int(row.row_id),
                "Drug": row.Drug,
                "Y": float(row.Y),
                "ConformerId": int(row.ConformerId),
                "conformer_method": row.conformer_method,
                "status": "ok",
                "error": "",
            })

        except Exception as exc:
            records.append({
                "row_id": int(row.row_id),
                "Drug": row.Drug,
                "Y": float(row.Y),
                "ConformerId": int(row.ConformerId),
                "conformer_method": row.conformer_method,
                "status": "failed",
                "error": repr(exc),
            })

        if processed_count % GRAPH_CHECKPOINT_EVERY == 0:
            _save_partial_graphs(graphs, records, partial_paths, manifest)

    # Keep a deterministic order by the original TDC row identifier.
    graphs.sort(key=lambda graph: int(graph.row_id))
    records_by_row = {int(record["row_id"]): record for record in records}

    graph_metadata_records = []
    for graph_idx, graph in enumerate(graphs):
        record = records_by_row[int(graph.row_id)]
        graph_metadata_records.append({
            "graph_idx": graph_idx,
            "row_id": int(graph.row_id),
            "Drug": record["Drug"],
            "Y": float(record["Y"]),
            "ConformerId": int(record["ConformerId"]),
            "conformer_method": record["conformer_method"],
            "num_atoms": int(graph.num_nodes),
            "num_edges": int(graph.edge_index.size(1)),
        })

    graph_metadata = pd.DataFrame(graph_metadata_records)
    failures = pd.DataFrame([
        record for record in records if record["status"] == "failed"
    ])

    torch.save(graphs, graph_path)
    graph_metadata.to_csv(metadata_path, index=False)
    failures.to_csv(failures_path, index=False)
    manifest_path.write_text(json.dumps(manifest, indent=2))

    for path in partial_paths:
        path.unlink(missing_ok=True)

    print(f"Saved {len(graphs)} graphs to {graph_path}")
    print(f"Graph build failures: {len(failures)}")

    return graphs, graph_metadata, failures

In [13]:
train_val_graphs, train_val_graph_metadata, train_val_graph_failures = (
    build_graphs_for_split(train_val_conformers, "train_val")
)

test_graphs, test_graph_metadata, test_graph_failures = (
    build_graphs_for_split(test_conformers, "test")
)

Building train_val graphs:   0%|          | 0/7961 [00:00<?, ?it/s]

Saved 7961 graphs to C:\Users\alexc\OneDrive\Desktop\proyectos\admet-ml\data\processed\aqsoldb\gnn\train_val_graphs.pt
Graph build failures: 0


Building test graphs:   0%|          | 0/1991 [00:00<?, ?it/s]

Saved 1991 graphs to C:\Users\alexc\OneDrive\Desktop\proyectos\admet-ml\data\processed\aqsoldb\gnn\test_graphs.pt
Graph build failures: 0


## 9. Inspect graph-generation results

In [14]:
summary = pd.DataFrame([
    {
        "split": "train_val",
        "source_molecules": len(train_val_conformers),
        "valid_conformers": int((
            train_val_conformers["ROMol3D"].notna()
            & train_val_conformers["ConformerId"].notna()
        ).sum()),
        "graphs": len(train_val_graphs),
        "graph_failures": len(train_val_graph_failures),
    },
    {
        "split": "test",
        "source_molecules": len(test_conformers),
        "valid_conformers": int((
            test_conformers["ROMol3D"].notna()
            & test_conformers["ConformerId"].notna()
        ).sum()),
        "graphs": len(test_graphs),
        "graph_failures": len(test_graph_failures),
    },
])

display(summary)
display(train_val_graph_metadata.head())

if len(train_val_graph_failures):
    display(train_val_graph_failures.head(20))
if len(test_graph_failures):
    display(test_graph_failures.head(20))

,split,source_molecules,valid_conformers,graphs,graph_failures
0,train_val,7985,7961,7961,0
1,test,1997,1991,1991,0


,graph_idx,row_id,Drug,Y,ConformerId,conformer_method,num_atoms,num_edges
0,0,0,O=Cc1ccc(Cl)cc1,-2.177078,0,MMFF,14,170
1,1,1,C=Cc1cccc(C)c1,-3.123150,0,MMFF,19,266
2,2,2,CN(C)c1ccc(C=O)cc1,-2.282769,0,MMFF,22,350
3,3,3,CC(=O)OC(C)(C)Cc1ccccc1,-2.394650,1,MMFF,30,606
4,4,4,COCCCCC(=O)c1ccc(C(F)(F)F)cc1,-3.544060,0,MMFF,33,566


In [15]:
def graph_statistics(graphs, split_name):
    return pd.Series({
        "split": split_name,
        "n_graphs": len(graphs),
        "mean_atoms": np.mean([graph.num_nodes for graph in graphs]),
        "median_atoms": np.median([graph.num_nodes for graph in graphs]),
        "mean_directed_edges": np.mean([
            graph.edge_index.size(1) for graph in graphs
        ]),
        "node_feature_dim": graphs[0].x.size(1),
        "edge_feature_dim": graphs[0].edge_attr.size(1),
    })


display(pd.DataFrame([
    graph_statistics(train_val_graphs, "train_val"),
    graph_statistics(test_graphs, "test"),
]))

,split,n_graphs,mean_atoms,median_atoms,mean_directed_edges,node_feature_dim,edge_feature_dim
0,train_val,7961,31.467529,27.0,663.976134,89,43
1,test,1991,36.903566,32.0,926.703164,89,43


## 10. Check the PyTorch Geometric batching

A `DataLoader` combines multiple molecular graphs into a disconnected batch. Node indices in `edge_index` are offset automatically by PyG. This is the format expected by the GINE message-passing layers.

In [16]:
BATCH_SIZE_CHECK = 4

sample_loader = DataLoader(
    train_val_graphs[:BATCH_SIZE_CHECK],
    batch_size=BATCH_SIZE_CHECK,
    shuffle=False,
)

batch = next(iter(sample_loader))

print(batch)
print("Number of molecules:", batch.num_graphs)
print("Total nodes:", batch.x.size(0))
print("Total edges:", batch.edge_index.size(1))
print("Node features:", batch.x.shape)
print("Edge features:", batch.edge_attr.shape)
print("Targets:", batch.y.shape)
print("Batch assignment:", batch.batch.shape)

assert batch.x.size(0) == batch.z.size(0) == batch.pos.size(0)
assert batch.edge_index.size(1) == batch.edge_attr.size(0)
assert batch.batch.size(0) == batch.x.size(0)
assert batch.y.numel() == batch.num_graphs
if batch.edge_index.numel():
    assert int(batch.edge_index.max()) < batch.num_nodes
assert torch.isfinite(batch.x).all()
assert torch.isfinite(batch.edge_attr).all()

print("PyG batching checks passed.")

DataBatch(x=[85, 89], edge_index=[2, 1392], edge_attr=[1392, 43], pos=[85, 3], z=[85], x_chem=[85, 17], x_dg=[85, 72], num_nodes=85, y=[4], smiles=[4], mol_id=[4], conf_id=[4], row_id=[4], dataset_split=[4], batch=[85], ptr=[5])
Number of molecules: 4
Total nodes: 85
Total edges: 1392
Node features: torch.Size([85, 89])
Edge features: torch.Size([1392, 43])
Targets: torch.Size([4])
Batch assignment: torch.Size([85])
PyG batching checks passed.


## 11. Save graph metadata and feature schema

The graph tensors are stored as PyTorch objects. Metadata is stored separately so that scaffold splits can later be mapped back to graphs using the original SMILES and row identifiers.

The graph cache contains raw, unnormalized features. `GraphFeatureNormalizer.fit()` must be called later on the training graphs for each scaffold split, then the fitted normalizer is applied to validation and test graphs.

In [17]:
feature_schema = get_graph_feature_names(config)
(GRAPH_DIR / "feature_schema.json").write_text(
    json.dumps(feature_schema, indent=2)
)

print("Saved graph files:")
for path in sorted(GRAPH_DIR.glob("*")):
    if path.is_file():
        print(f"{path.name:<45} {path.stat().st_size / (1024**2):8.2f} MB")

Saved graph files:
feature_schema.json                               0.01 MB
test_graph_failures.csv                           0.00 MB
test_graph_manifest.json                          0.01 MB
test_graph_metadata.csv                           0.14 MB
test_graphs.pt                                  386.75 MB
train_val_graph_failures.csv                      0.00 MB
train_val_graph_manifest.json                     0.01 MB
train_val_graph_metadata.csv                      0.51 MB
train_val_graphs.pt                            1141.85 MB


## Conclusions

This notebook creates one PyTorch Geometric `Data` object per molecule, preserving the official TDC `train_val` and `test` partitions.

Each graph contains:

- `z`: atomic numbers;
- `x_chem`: chemical atom features;
- `x_dg`: atom-centred G6 density and curvature features;
- `x`: concatenated chemical and G6 node features;
- `pos`: selected conformer coordinates;
- `edge_index`: directed molecular/spatial connectivity;
- `edge_attr`: bond and geometric edge features;
- `y`: experimental solubility;
- `smiles`, `mol_id`, `row_id`: identifiers for traceability.

The next notebook should create the scaffold-aware train/validation split, fit `GraphFeatureNormalizer` on the training fold only, and train the GINE model.